In [1]:
import tomotopy as tp
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from operator import itemgetter
import matplotlib.pyplot as plt
import pandas as pd
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import re
import openai
from googletrans import Translator
import matplotlib.pyplot as plt
import operator

In [2]:
def get_topics(line):
    line=line.replace('\'','')
    line=line.split("[")[1]
    line=line.split("]")[0]
    return line.split(", ")

In [3]:
def get_word_embedding_based_similarity(topic1, topic2):
    similarity = 0
    panalty = 0
    for i in topic_dict.get(topic1)[:10]:
        for j in topic_dict.get(topic2)[:10]:
            s = word2vec_model.wv.similarity(i,j)
            similarity+= s
            if s == 1:
                panalty +=1
    if panalty > 5 : similarity /=2
    return similarity/(10*10)

In [4]:
def get_similarity(topn_dict):
    similarity = 0
    cnt = 0
    for d1_topic in topn_dict:
        similarity+=get_word_embedding_based_similarity(0,d1_topic)
        cnt+=1
        for d2_topic in topn_dict.get(d1_topic):
            similarity+=get_word_embedding_based_similarity(d1_topic, d2_topic)
            cnt+=1
    s = similarity/cnt
    return s

In [5]:
def get_diveristy(topn_dict):
    cnt = len(topn_dict)
    diversity = 0
    for d1_topic in topn_dict:
        diversity += (1-get_word_embedding_based_similarity(0,d1_topic))
        
        if len(topn_dict.get(d1_topic)) == 1: continue
        else : d1_topic_list = topn_dict.get(d1_topic)
        for i in range(len(d1_topic_list)):
            for j in range(i):
                diversity += (1 - get_word_embedding_based_similarity(d1_topic_list[i],d1_topic_list[j]))
                cnt+=1
    return diversity / cnt

In [6]:
def assign_topic(doc, topics):
    # 각 토픽에 대한 확률을 저장할 딕셔너리
    topic_probabilities = {}

    # 각 토픽에 대해
    for topic_id, topic_words in topics.items():
        # 해당 토픽에 속한 단어들의 우선순위를 저장
        topic_word_priority = {word: priority for priority, word in enumerate(reversed(topic_words))}

        # 현재 문서에서 해당 토픽에 속한 단어들만 추출
        doc_words_in_topic = [word for word in doc if word in topic_word_priority]

        # 각 단어에 대한 우선순위의 합을 계산
        priority_sum = sum(topic_word_priority[word] for word in doc_words_in_topic)

        # 해당 토픽에 속할 확률을 딕셔너리에 저장
        topic_probabilities[topic_id] = priority_sum

    # 확률이 가장 높은 토픽을 선택
    most_probable_topic = max(topic_probabilities.items(), key=operator.itemgetter(1))[0]

    return most_probable_topic

In [7]:
def find_parent(dictionary, value):
    for key, values in dictionary.items():
        if value in values:
            return key
    return None

In [8]:
top_n = 10

In [11]:
word2vec_model = Word2Vec.load('./models/word2vec.model')

In [12]:
score_df = pd.DataFrame(columns=['phase','k','hcsd'])
for phase in [1,2,3,4]:
    raw = pd.read_feather(f'./Datasets/covid{phase}.ftr')
    texts =raw.okt
    raw['label'] = range(len(raw))
    dictionary = Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]

    file=open(f'./result/hierarchical_struture_{phase}.txt', 'r')
    data = file.readlines()


    num_depth0 = 0
    num_depth1 = 0
    num_depth2 = 0
    for line in data:
        if '\t' not in line: num_depth0+=1
        elif '\t\t'not in line: num_depth1+=1
        else: num_depth2+=1
    root = num_depth0
    num_depth1 += num_depth0
    num_depth2 += num_depth2


    topic_dict={}
    parent_dict={}

    for line in data:
        # depth0
        if '\t' not in line:
            num_depth0 -=1
            topic_dict[num_depth0] = get_topics(line)

        # depth1
        elif '\t\t'not in line:
            num_depth1 -=1
            topic_dict[num_depth1] = get_topics(line)
            if parent_dict.get(num_depth0) is None :
                parent_dict[num_depth0] = [num_depth1]
            else :
                temp = parent_dict.get(num_depth0)
                temp.append(num_depth1)
                parent_dict[num_depth0] = temp

        # depth2
        else:
            num_depth2 -=1
            topic_dict[num_depth2] = get_topics(line)
            if parent_dict.get(num_depth1) is None :
                parent_dict[num_depth1] = [num_depth2]
            else :
                temp = parent_dict.get(num_depth1)
                temp.append(num_depth2)
                parent_dict[num_depth1] = temp

    leaf_topics = {}
    for i in range(root):
        if parent_dict.get(i) is not None:
            for d1_topic in parent_dict.get(i):
                if parent_dict.get(d1_topic) is not None:
                    for d2_topic in parent_dict.get(d1_topic):
                        leaf_topics[d2_topic] = topic_dict.get(d2_topic)

    for i in range(len(raw)):
        raw.label[i] = assign_topic(raw.okt[i],leaf_topics)


    #similarity_list = []
    #diversity_list=[]
    #coherence_list=[]
    #hcsd_list=[]
    for topic_num in range(10,55,5):
        res = raw['label'].value_counts()[:topic_num].index
        topn_dict = {}
        for child in res:
            if find_parent(parent_dict, child) in topn_dict:
                temp = topn_dict.get(find_parent(parent_dict, child))
                temp.append(child)
                topn_dict[find_parent(parent_dict, child)] = temp
            else : topn_dict[find_parent(parent_dict, child)] = [child]  


        topics = []
        #topics.append([word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
        for d1_topic in topn_dict:
            topics.append(topic_dict.get(d1_topic)[:topic_num])
            for d2_topic in topn_dict.get(d1_topic):
                topics.append(topic_dict.get(d2_topic)[:topic_num])

        cm = CoherenceModel(topics = topics,
               #corpus = corpus,
               dictionary = dictionary,
               texts = texts,
               coherence = 'c_npmi')
        # get proposed score

        similarity=get_similarity(topn_dict)
        #similarity_list.append(similarity)

        diversity=get_diveristy(topn_dict)
        #diversity_list.append(diversity)

        coherence =cm.get_coherence()
        # coherence nomalization
        coherence = (coherence +1)/2
        #coherence_list.append(coherence)

        score = (similarity+diversity+coherence)/3
        #hcsd_list.append(score)
        score_df = score_df.append({'phase':phase,'k':topic_num,'hcsd':score},ignore_index=True)
        print("phase",phase,", topic num:",topic_num,similarity,diversity,coherence,score)
    print()

/home/deallab/anaconda3/envs/harin/lib/python3.7/site-packages/ipykernel_launcher.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


phase 1 , topic num: 10 0.6161789334776174 0.5240294961454348 0.4053971042972009 0.5152018446400843
phase 1 , topic num: 15 0.6424917444425564 0.4776188888829201 0.3897984449800258 0.5033030261018341
phase 1 , topic num: 20 0.6507052833531054 0.450269436019007 0.3759178990460565 0.492297539472723
phase 1 , topic num: 25 0.6595853649050268 0.4423197494271314 0.3780748739780809 0.4933266627700797
phase 1 , topic num: 30 0.6700947307055816 0.4473883045685943 0.3756127985540445 0.49769861127607345
phase 1 , topic num: 35 0.6743580511138824 0.4421317857551651 0.37201044107709125 0.49616675931537957
phase 1 , topic num: 40 0.6677747724422565 0.4372163819331438 0.3722029679248656 0.49239804076675525
phase 1 , topic num: 45 0.6725287256609864 0.4244925028660842 0.3709448816120214 0.48932203671303065
phase 1 , topic num: 50 0.6795738253755935 0.41531954420759143 0.368495533503025 0.48779630102873667

phase 2 , topic num: 10 0.5630317844564302 0.5812843549831305 0.42624995868628235 0.52352203270

In [ ]:
score_df

In [ ]:
score_df.to_feather('./result/cluhtm_result.ftr')

In [ ]:
top_n=10

topics = []
#print("topic0 : ",[word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
#print()
for d1_topic in topn_dict:
    print("  topic",d1_topic, ": ",topic_dict.get(d1_topic)[:10])
    for d2_topic in topn_dict.get(d1_topic):
        print("    topic",d2_topic, ": ",topic_dict.get(d2_topic)[:10])
    print()
